# Pipeline Component Testing
Tests each stage of the filing pipeline independently:  
**Loader → Chunker → (Sentiment/Risk → Summarizer in later phases)**

Run cells top-to-bottom. Each section is self-contained.

## 0 · Setup

In [ ]:
import sys, os, pprint

# Make app/ importable from the notebooks/ directory
REPO_ROOT = os.path.abspath(os.path.join(os.getcwd(), '..'))
if REPO_ROOT not in sys.path:
    sys.path.insert(0, REPO_ROOT)

print('Repo root:', REPO_ROOT)
print('Python   :', sys.executable)

---
## 1 · Loader — Markdown (Synthetic Corpus)
Tests `MarkdownFilingLoader` on the existing synthetic `.md` filings.

In [ ]:
from app.loader import MarkdownFilingLoader

MARKDOWN_CORPUS_DIR = os.path.join(REPO_ROOT, 'corpus', 'filings')
md_loader = MarkdownFilingLoader(corpus_dir=MARKDOWN_CORPUS_DIR)

# ── Change this to any filing in corpus/filings/ ──
FILING_ID = 'HLSR-2024'

md_docs = md_loader.load(FILING_ID)

print(f'Filing  : {FILING_ID}')
print(f'Sections loaded: {len(md_docs)}')
print()
for i, doc in enumerate(md_docs):
    print(f"  [{i}] section='{doc['metadata']['section']}'  chars={len(doc['page_content'])}")

In [ ]:
# Inspect a specific section's content
INSPECT_INDEX = 0   # change to any index from the list above

doc = md_docs[INSPECT_INDEX]
print(f"Section : {doc['metadata']['section']}")
print(f"Source  : {doc['metadata']['source']}")
print('-' * 60)
print(doc['page_content'][:800], '...' if len(doc['page_content']) > 800 else '')

---
## 2 · Loader — PDF (Real 10-K / 10-Q)

`PDFFilingLoader` uses **pdfplumber** to extract prose + tables.  
Tables are converted to Markdown pipe format and appended under a `[TABLES]` marker.

> **Set `PDF_PATH` below** to the absolute path of any 10-K or 10-Q PDF.

In [ ]:
# ╔══════════════════════════════════════════════════════════╗
# ║  CONFIGURE: set the path to your PDF and a filing ID    ║
# ╚══════════════════════════════════════════════════════════╝

PDF_PATH    = r'C:\path\to\your\10K.pdf'   # <── change this
PDF_FILING_ID = 'MY-COMPANY-2024'          # <── change this (used as title in metadata)

# Derive PDF_DIR and filename stem for the loader
PDF_DIR  = os.path.dirname(PDF_PATH)
PDF_STEM = os.path.splitext(os.path.basename(PDF_PATH))[0]  # filename without .pdf

print(f'PDF dir  : {PDF_DIR}')
print(f'Stem     : {PDF_STEM}')
print(f'Exists   : {os.path.exists(PDF_PATH)}')

In [ ]:
from app.loader import PDFFilingLoader

pdf_loader = PDFFilingLoader(pdf_dir=PDF_DIR)

# Load using the stem as filing_id (loader appends .pdf automatically)
pdf_docs = pdf_loader.load(PDF_STEM)

print(f'Pages extracted : {len(pdf_docs)}')
pages_with_tables = [d for d in pdf_docs if d['metadata'].get('has_tables')]
print(f'Pages with tables: {len(pages_with_tables)}')
print()
for i, doc in enumerate(pdf_docs[:10]):   # show first 10 pages
    m = doc['metadata']
    print(f"  [page {m['page']:>3}] section='{m['section']}'  has_tables={m['has_tables']}  chars={len(doc['page_content'])}")

In [ ]:
# Inspect a page — view prose + extracted table side by side
INSPECT_PAGE = 1   # 1-indexed page number

target = next((d for d in pdf_docs if d['metadata']['page'] == INSPECT_PAGE), None)
if target is None:
    print(f'Page {INSPECT_PAGE} not found in extracted docs.')
else:
    m = target['metadata']
    print(f"Page     : {m['page']}")
    print(f"Section  : {m['section']}")
    print(f"Has tables: {m['has_tables']}")
    print('=' * 60)
    content = target['page_content']
    # Split prose from tables for readability
    if '[TABLES]' in content:
        prose_part, table_part = content.split('[TABLES]', 1)
        print('--- PROSE ---')
        print(prose_part[:600])
        print('\n--- TABLES (Markdown) ---')
        print(table_part[:800])
    else:
        print(content[:1000])

In [ ]:
# Section coverage summary across entire PDF
from collections import Counter

section_counts = Counter(d['metadata']['section'] for d in pdf_docs)
print('Section distribution (pages per section):')
for section, count in section_counts.most_common():
    print(f'  {section:<35} {count} page(s)')

---
## 3 · Chunker
Tests `chunk_documents` on the Markdown docs (works identically for PDF docs).

Uses `RecursiveCharacterTextSplitter` if `langchain-text-splitters` is installed,  
otherwise falls back to the original fixed-size splitter.

In [ ]:
from app.chunker import chunk_documents, _LANGCHAIN_AVAILABLE

print(f'LangChain splitter available: {_LANGCHAIN_AVAILABLE}')

CHUNK_SIZE    = 1000
CHUNK_OVERLAP = 150

md_chunks = chunk_documents(md_docs, chunk_size=CHUNK_SIZE, chunk_overlap=CHUNK_OVERLAP)

print(f'\nMarkdown — sections in: {len(md_docs)}  →  chunks out: {len(md_chunks)}')

text_chunks  = [c for c in md_chunks if c['metadata'].get('chunk_type') != 'table']
table_chunks = [c for c in md_chunks if c['metadata'].get('chunk_type') == 'table']
print(f'  text chunks : {len(text_chunks)}')
print(f'  table chunks: {len(table_chunks)}')

sizes = [len(c['page_content']) for c in md_chunks]
print(f'\nChunk sizes — min: {min(sizes)}  max: {max(sizes)}  avg: {sum(sizes)//len(sizes)}')

In [ ]:
# Inspect individual chunks
INSPECT_CHUNK = 0   # change to any chunk index

ch = md_chunks[INSPECT_CHUNK]
print(f"Section    : {ch['metadata'].get('section')}")
print(f"Chunk index: {ch['metadata'].get('chunk_index')}")
print(f"Chunk type : {ch['metadata'].get('chunk_type', 'text')}")
print(f"Chars      : {len(ch['page_content'])}")
print('-' * 60)
print(ch['page_content'])

In [ ]:
# Overlap sanity check — confirm adjacent text chunks share content
text_only = [c for c in md_chunks if c['metadata'].get('chunk_type', 'text') == 'text']

if len(text_only) >= 2 and _LANGCHAIN_AVAILABLE:
    a = text_only[0]['page_content']
    b = text_only[1]['page_content']
    # Count shared tokens
    tokens_a = set(a.lower().split())
    tokens_b = set(b.lower().split())
    overlap_tokens = tokens_a & tokens_b
    print(f'Shared tokens between chunk 0 and chunk 1: {len(overlap_tokens)}')
    print(f'Sample overlap tokens: {list(overlap_tokens)[:10]}')
else:
    print('Not enough text chunks or LangChain unavailable — overlap check skipped.')

---
## 4 · Chunker on PDF Docs
Same chunker, now applied to PDF-extracted documents.  
Table chunks should appear as intact Markdown tables (not split mid-row).

In [ ]:
# Skip this cell if PDF was not loaded above
if 'pdf_docs' not in dir() or not pdf_docs:
    print('No PDF docs loaded — run Section 2 first.')
else:
    pdf_chunks = chunk_documents(pdf_docs, chunk_size=CHUNK_SIZE, chunk_overlap=CHUNK_OVERLAP)

    pdf_text_chunks  = [c for c in pdf_chunks if c['metadata'].get('chunk_type') != 'table']
    pdf_table_chunks = [c for c in pdf_chunks if c['metadata'].get('chunk_type') == 'table']

    print(f'PDF — pages in: {len(pdf_docs)}  →  chunks out: {len(pdf_chunks)}')
    print(f'  text chunks : {len(pdf_text_chunks)}')
    print(f'  table chunks: {len(pdf_table_chunks)}')

    if pdf_table_chunks:
        print('\n--- First table chunk (Markdown pipe format) ---')
        print(pdf_table_chunks[0]['page_content'][:600])

---
## 5 · LoaderFactory — Dynamic Source Switching
Tests the factory toggle between `markdown` and `pdf` sources.

In [ ]:
from app.loader import LoaderFactory

# Test markdown path via factory
os.environ['FILING_SOURCE'] = 'markdown'
loader = LoaderFactory.get()
print(f'Factory returned: {type(loader).__name__}')

factory_docs = loader.load('HLSR-2024')
print(f'Sections loaded : {len(factory_docs)}')
assert all('section' in d['metadata'] for d in factory_docs), 'metadata.section missing!'
assert all('source'  in d['metadata'] for d in factory_docs), 'metadata.source missing!'
print('✓ All documents have required metadata keys (section, source)')

In [ ]:
# Test pdf path via factory (expects PDF_DIR env var + PDF file present)
if 'pdf_docs' in dir() and pdf_docs:
    os.environ['FILING_SOURCE'] = 'pdf'
    os.environ['PDF_DIR'] = PDF_DIR
    pdf_factory_loader = LoaderFactory.get()
    print(f'Factory returned: {type(pdf_factory_loader).__name__}')
    pdf_factory_docs = pdf_factory_loader.load(PDF_STEM)
    print(f'Pages loaded via factory: {len(pdf_factory_docs)}')
    print('✓ PDFFilingLoader works through factory')
else:
    print('PDF not loaded — skipping factory PDF test.')

# Reset to default
os.environ['FILING_SOURCE'] = 'markdown'